# Train ResNet-56 on CIFAR-10

We’ll turn the architecture from [Notebook 1](01_paper_and_model.ipynb) into a trained image classifier. We’ll inspect a batch, follow one optimizer update, and then use the training helper for the full run. The evaluation notebook will use the saved model and metric history.

## 1. Set up the run

The model, data loaders, and training loop live in `src/`. The seed controls the train/validation split, the training shuffle, and the initial weights. We use CUDA when PyTorch detects a supported GPU. The data loader starts with zero worker processes so it can run in a Windows notebook kernel. The code uses the local CIFAR-10 files when available and can reuse the download from the Highway Networks project.

In [3]:
import sys
from itertools import islice
from pathlib import Path
from time import perf_counter

import torch

cwd = Path.cwd()
project = next(
    path for path in (cwd, *cwd.parents, cwd / "resnet-56")
    if (path / "notebooks" / "02_train_resnet56.ipynb").is_file()
)
sys.path.insert(0, str(project))

from src.data import get_cifar10
from src.model import ResNet56
from src.training import TrainingConfig, load_checkpoint, make_optimizer, run_epoch, train_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed, batch_size = 42, 128
config = TrainingConfig()
torch.manual_seed(seed)

data_root = project / "data"
shared_root = project.parent / "highway-networks" / "data"
if not (data_root / "cifar-10-batches-py").is_dir() and (shared_root / "cifar-10-batches-py").is_dir():
    data_root = shared_root

print("Device:", torch.cuda.get_device_name(device) if device.type == "cuda" else device)
print("Dataset directory:", data_root)

Device: NVIDIA GeForce RTX 4050 Laptop GPU
Dataset directory: c:\Users\rbern\Desktop\DL\highway-networks\data


## 2. Prepare CIFAR-10 batches

A seeded split gives us **45,000 training images and 5,000 validation images**. The optimizer learns from the training images. Validation measures how the model performs on unseen images during the run and determines which checkpoint to keep. The paper used a 45,000/5,000 split to choose the training schedule. We keep the split throughout this run and reserve the official test set for Notebook 3. [ResNet paper, §4.2](https://arxiv.org/pdf/1512.03385#page=7)

Each image starts as a `(3, 32, 32)` tensor scaled to `[0, 1]`. We subtract a mean image computed pixel by pixel from the training split, then apply a random crop with four pixels of padding and a random horizontal flip to training images. Those transformations expose the model to small shifts and reflections. Validation uses the same mean image and the original views.

A `DataLoader` stacks images into `(batch, 3, 32, 32)` tensors and pairs them with one class index per image. The training loader reshuffles each pass through the data. Validation uses a fixed order.

In [4]:
data = get_cifar10(data_root, device, seed=seed, batch_size=batch_size)
smoke_batches = iter(data.make_train_loader())
images, labels = next(smoke_batches)

print("Image and label shapes:", tuple(images.shape), tuple(labels.shape))
print("First few labels:", [data.class_names[i] for i in labels[:5].tolist()])
print("Validation images:", len(data.val_loader.dataset))

Image and label shapes: (128, 3, 32, 32) (128,)
First few labels: ['cat', 'automobile', 'frog', 'cat', 'ship']
Validation images: 5000


## 3. Follow one optimizer update

1. **Forward pass.** ResNet-56 produces ten logits per image. Cross-entropy uses the logits and the correct class index to measure how much probability the model assigns to that class.
2. **Backward pass.** Clear gradients from the previous update and differentiate the loss with respect to the current weights.
3. **SGD update.** Move the weights using the gradients and the learning rate. Momentum smooths updates using a running combination of recent gradients. Weight decay discourages large weights.

In training mode, batch normalization uses the current batch to normalize each channel and updates its running statistics. During validation, evaluation mode uses those stored statistics so a prediction’s normalization is consistent across batches.

In [5]:
torch.manual_seed(seed)
smoke_model = ResNet56().to(device)
smoke_model.train()
optimizer = make_optimizer(smoke_model, config)

weight = next(smoke_model.parameters())
initial_weight = weight.detach().clone()
optimizer.zero_grad(set_to_none=True)
logits = smoke_model(images.to(device))
loss = torch.nn.functional.cross_entropy(logits, labels.to(device))

assert logits.shape == (len(labels), 10) and torch.isfinite(loss).item()
loss.backward()
optimizer.step()
assert not torch.equal(initial_weight, weight.detach())

print(f"Logits: {tuple(logits.shape)} | First batch loss: {loss.item():.4f}")
print("Verified that SGD updated the stem weights.")

Logits: (128, 10) | First batch loss: 3.3024
Verified that SGD updated the stem weights.


## 4. Use the paper’s learning-rate schedule

The run uses batch size **128**, SGD momentum **0.9**, and weight decay **0.0001**. An **update** is one optimizer step on one batch. An **epoch** is a pass through the training loader. The schedule counts updates, so a learning-rate change can happen partway through an epoch. The larger initial rate lets the model make bigger moves early in training. The smaller rates later let it refine the weights. [ResNet paper, §4.2](https://arxiv.org/pdf/1512.03385#page=7)

| Completed updates | Learning rate for the next update |
| ---: | ---: |
| 0 | 0.1 |
| 32,000 | 0.01 |
| 48,000 | 0.001 |

Training stops after **64,000 updates**. The helper ends the final epoch as soon as it reaches that count.

## 5. Run a short smoke test

Use the same batch size and training loop for ten more updates, then run a few validation batches. This checks that the loader, model, optimizer, and evaluation mode work together. Early validation loss can fluctuate sharply while the weights and batch normalization statistics are still changing. Timing the updates gives us a rough training-only estimate. Validation and checkpoint writing add to that time.

In [6]:
preview_steps = 10
started = perf_counter()
train_loss, _ = run_epoch(smoke_model, islice(smoke_batches, preview_steps), device, optimizer)
seconds_per_step = (perf_counter() - started) / preview_steps
val_loss, val_accuracy = run_epoch(smoke_model, islice(data.val_loader, 2), device)

print(f"Preview train loss: {train_loss:.4f} | Validation loss: {val_loss:.4f}")
print(f"Preview validation accuracy: {val_accuracy:.2f}%")
print(f"Estimated training time before validation: {seconds_per_step * config.total_steps / 3600:.1f} h")

Preview train loss: 5.9193 | Validation loss: 8002.1455
Preview validation accuracy: 7.03%
Estimated training time before validation: 1.6 h


## 6. Train and save the results

Set `RUN_MODE` to `"start"` to start the full run with freshly initialized weights. For an interrupted run with a saved checkpoint, set it to `"resume"`. Each completed epoch records sample-weighted loss and accuracy for training and validation.

- `latest.pth` saves the current model, optimizer, scheduler, and update count after validation so we can resume from that point.
- `best.pth` saves the model from the highest-validation-accuracy epoch so Notebook 3 can evaluate it. The metric history in `latest.pth` covers the full run and will be used for learning curves.

Both files also store the seed, settings, and training mean image. The final lines reload the best model to check that the checkpoint is usable.

In [ ]:
RUN_MODE = "start"  # Set to "start" or "resume" to run full training.
checkpoint_dir = project / "checkpoints" / "sgd-paper"

if RUN_MODE in ("start", "resume"):
    history, best_path = train_model(
        data, device, checkpoint_dir, config, resume=RUN_MODE == "resume",
    )
    _, best_checkpoint = load_checkpoint(best_path, device)
    print(f"Best validation accuracy: {best_checkpoint['best_val_accuracy']:.2f}%")
    print("Best checkpoint:", best_path)
else:
    print("Smoke test complete. Set RUN_MODE when ready for the full run.")

Smoke test complete. Set RUN_MODE when ready for the full run.
